# 🔍 Data Quality Exploration — Automotive Dataset

Exploratory analysis that drove the cleaning decisions in the main pipeline
(`cars_medallion_pipeline`).

This notebook documents **how** each data anomaly was discovered and **why**
each decision was made. It reads from the Silver table produced by the main
pipeline — run that notebook first.

**Guiding principle:** exclude data that is *impossible or internally contradictory*,
never data that is merely *surprising*. A surprising-but-real value is an insight,
not an error.

### Investigation 1 — The Mazda price outlier

`brand_metrics` showed Mazda with a max price of **$5,000,000** — implausible
for a mainstream brand. Drilling in revealed the **Mazda 787B**, a Le Mans
race prototype rather than a showroom model.

→ This discovery motivated the `is_showroom_model` flag in Silver.

In [0]:
spark.sql("""
SELECT car_name, price_usd, horsepower, fuel_category
FROM workspace.silver.clean_cars
WHERE company_name = 'MAZDA'
ORDER BY price_usd DESC
LIMIT 5
""").display()

car_name,price_usd,horsepower,fuel_category
787B (Race Car),5000000,700,Petrol
CX-90,50000,340,Petrol
CX-80,45000,250,Petrol
CX-60,40000,187,Diesel
Eunos Cosmo,40000,280,Petrol


### Investigation 2 — Verifying the showroom flag

Confirms `is_showroom_model` correctly marks non-showroom vehicles
(race cars, prototypes) — while keeping them in Silver rather than deleting them.
Expected: Mazda 787B (Race Car) and VW XL1 (prototype).

In [0]:
spark.sql("""
SELECT car_name, company_name, is_showroom_model
FROM workspace.silver.clean_cars
WHERE is_showroom_model = false
""").display()

car_name,company_name,is_showroom_model
Volkswagen XL1,VOLKSWAGEN,false
787B (Race Car),MAZDA,false


### Investigation 3 — Impossible horsepower values

The power sweet-spot analysis showed a max of **2488 hp** in the Economy tier.
This query isolates out-of-range values.

**Culprit:** a **Nissan Urvan** (a commercial van) recorded at 2488 hp —
almost certainly its 2488**cc** engine displacement mis-entered into the
horsepower column. The Bugatti models (1600–1850 hp) were confirmed as
**real** hypercars and kept.

→ Motivated the validity filter `horsepower <= 2000` in the Gold analyses.


In [0]:
spark.sql("""
SELECT company_name, car_name, price_usd, horsepower, price_segment, fuel_category
FROM workspace.silver.clean_cars
WHERE is_showroom_model = true
  AND (horsepower > 1500 OR horsepower < 40)
ORDER BY horsepower DESC
""").display()

company_name,car_name,price_usd,horsepower,price_segment,fuel_category
NISSAN,Urvan,28000,2488,Economy,Diesel
BUGATTI,Bolide,4500000,1850,Luxury,Petrol
BUGATTI,Mistral,5000000,1600,Luxury,Petrol
BUGATTI,Chiron Super Sport,3500000,1600,Luxury,Petrol
BUGATTI,Centodieci,9000000,1600,Luxury,Petrol
TATA MOTORS,Nano GenX,4000,37,Economy,Petrol
VOLKSWAGEN,Karmann Ghia,25000,34,Economy,Petrol
MAZDA,Carol P360,8000,26,Economy,Petrol


### Investigation 4 — Are the high-hp affordable cars real?

Before trusting the data, checked whether 300+ hp Economy cars and 650+ hp
Premium cars were errors or genuine.

**Result: all real** — Corvette, Mustang, hot hatches. High performance-per-dollar
is a genuine and interesting market feature, so these rows were deliberately **kept**.
This is the counter-example to over-aggressive cleaning: surprising ≠ wrong.

In [0]:
spark.sql("""
SELECT company_name, car_name, price_usd, horsepower, price_segment
FROM workspace.silver.clean_cars
WHERE is_showroom_model = true
  AND fuel_category = 'Petrol'
  AND vehicle_class = 'light'
  AND (
    (price_segment = 'Economy' AND horsepower > 300)
    OR (price_segment = 'Premium' AND horsepower > 650)
  )
ORDER BY price_segment, horsepower DESC
""").display()

company_name,car_name,price_usd,horsepower,price_segment
TOYOTA,TUNDRA,39900,381,Economy
GMC,Sierra 1500,37100,355,Economy
NISSAN,370Z,30000,332,Economy
NISSAN,FRONTIER,30000,310,Economy
GMC,Canyon,36900,310,Economy
CHEVROLET,Traverse,35915,310,Economy
CHEVROLET,Colorado,30695,310,Economy
CHEVROLET,Colorado WT,29200,310,Economy
CHEVROLET,Traverse LT,37200,310,Economy
BMW,M135i XDRIVE,30000,302,Economy


### Investigation 5 — The GMC "luxury but slow" case

GMC appeared in the Luxury tier with an average top speed of only 170 km/h,
far below other luxury brands. Investigation confirmed this is **valid data**:
GMC builds large, expensive SUVs and pickups (Yukon, Hummer) that are costly
enough for the Luxury tier but aerodynamically limited in top speed.

→ Kept as-is. Documents a known **limitation of price-based segmentation**:
classifying "Luxury" by price alone groups a Ferrari with a luxury pickup.

In [0]:
spark.sql("""
SELECT car_name, price_usd, horsepower, top_speed_kmh, price_segment
FROM workspace.silver.clean_cars
WHERE company_name = 'GMC'
ORDER BY price_usd DESC
""").display()

car_name,price_usd,horsepower,top_speed_kmh,price_segment
Hummer EV Omega Edition,112700,1000,170,Luxury
Hummer EV SUV Extreme Off-Road,110000,830,170,Luxury
Hummer EV Pickup,108700,625,170,Luxury
Sierra EV Denali,108700,754,170,Luxury
Hummer EV SUV,105600,625,170,Luxury
Hummer EV Adventure Series,105000,830,170,Luxury
Hummer EV SUV Edition 1,96550,830,170,Premium
Hummer EV Pickup Edition 1,96550,1000,170,Premium
Yukon XL Denali Ultimate,93800,420,200,Premium
Yukon Denali Ultimate Black,90000,420,210,Premium
